# 西洋棋子偵測：在 Kaggle 上微調 YOLO11

這是小試跑，不是手機照片的預設流程（預設仍是 `yolo11m`），也不是 COCO。

資料是公開的 [LibreYOLO/chess-pieces-mjzgj](https://huggingface.co/datasets/LibreYOLO/chess-pieces-mjzgj)。Hub 授權欄位和快照裡的 `roboflow.license` 都是 **CC BY 4.0**。上游是 [Roboflow 100 的同一份資料](https://universe.roboflow.com/roboflow-100/chess-pieces-mjzgj/dataset/1)。不需要 Kaggle、Roboflow 或 Hugging Face 帳號。

2026-11-15 對公開快照點過數：train **202**、valid **58**、test **29**。圖是 640×640 JPEG。標註是一般 YOLO 框。類別 13 個：12 個是黑白棋子，第 0 類 `bishop` 在來源裡只有 1 個框。不要重排編號。

**預設是微調**官方權重。`MODE = "finetune"` 且 `MODEL_SIZE = "x"`，也就是 `yolo11x.pt`。`n` / `s` / `m` / `l` 可以改。從零訓練才把 `MODE` 改成 `"scratch"`。

停止條件沿用 `patience` 和 `AcceptableResultStop`。權重寫到 `/kaggle/working`。

## English

Public CC BY 4.0 chess-piece set, 202/58/29 images, 13 classes. Default is fine-tuning official `yolo11x.pt`. Change `MODEL_SIZE` to use n, s, m, or l. `MODE = "scratch"` trains the matching yaml from random weights. No account is required to download the dataset.


In [ ]:
# =============================================================================
# SWITCH. Default is fine-tuning official YOLO11x weights.
# "finetune" -> yolo11{size}.pt
# "scratch"  -> yolo11{size}.yaml
# MODEL_SIZE is n, s, m, l, or x. Kaggle default is x.
# =============================================================================
MODE = "finetune"
MODEL_SIZE = "x"

GPU = "t4x2"  # "t4x2" or "p100"

# Ultralytics patience. Fitness in Ultralytics 8.4 is validation mAP50-95.
PATIENCE = 15
# Total val loss is box + cls + dfl. MAP_TARGET is mAP50-95 for this pilot,
# not the published COCO number. None disables the target.
LOSS_MIN_DELTA = 0.01
LOSS_PATIENCE = 15
MAP_TARGET = 0.50

TIME_HOURS = 11.0
SAVE_PERIOD = 5
IMGSZ = 640
WORKERS = 2
AMP = True

HUB_REPO = "LibreYOLO/chess-pieces-mjzgj"
DATASET_DIR = "/kaggle/temp/chess"
RESUME_LAST = ""  # e.g. "/kaggle/input/chess-yolo11-ckpt/last.pt"
REPO_ROOT = "/kaggle/working/object-recognition"
PROJECT = "/kaggle/working/chess-yolo11"

if MODEL_SIZE not in {"n", "s", "m", "l", "x"}:
    raise ValueError("MODEL_SIZE must be n, s, m, l, or x")
if MODE == "finetune":
    MODEL = f"yolo11{MODEL_SIZE}.pt"
    PRETRAINED = True
    EPOCHS = 80
elif MODE == "scratch":
    MODEL = f"yolo11{MODEL_SIZE}.yaml"
    PRETRAINED = False
    EPOCHS = 150
else:
    raise ValueError('MODE must be "finetune" (default) or "scratch"')

if GPU == "t4x2":
    DEVICE = "0,1"
    # Safe for yolo11x on two 16 GB cards (4 images each). Smaller sizes can go up.
    BATCH = 8
elif GPU == "p100":
    DEVICE = "0"
    BATCH = 4
else:
    raise ValueError('GPU must be "t4x2" or "p100"')

print(
    f"MODE={MODE} model={MODEL} device={DEVICE} batch={BATCH} "
    f"epochs={EPOCHS} patience={PATIENCE} map_target={MAP_TARGET}"
)
print("Dataset is 289 images. The COCO hours-per-epoch table does not apply.")
print("Estimate, not a measurement: one yolo11x fine-tune should fit in a single 12-hour T4 x2 session.")


In [ ]:
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("huggingface_hub") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "huggingface_hub"])
if importlib.util.find_spec("ultralytics") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "ultralytics>=8.3"])
print("Download helpers are installed. No API token is used.")


In [ ]:
from pathlib import Path

import yaml
from huggingface_hub import snapshot_download

dataset_dir = Path(DATASET_DIR)
dataset_dir.mkdir(parents=True, exist_ok=True)
snapshot_download(repo_id=HUB_REPO, repo_type="dataset", local_dir=str(dataset_dir))

counts = {}
for split in ("train", "valid", "test"):
    images = [p for p in (dataset_dir / split / "images").glob("*") if p.suffix.lower() in {".jpg", ".jpeg", ".png"}]
    counts[split] = len(images)
print("Image counts:", counts)
if counts != {"train": 202, "valid": 58, "test": 29}:
    raise RuntimeError(f"Unexpected image counts: {counts}")

data_yaml = dataset_dir / "data.yaml"
loaded = yaml.safe_load(data_yaml.read_text(encoding="utf-8"))
license_name = str((loaded.get("roboflow") or {}).get("license", ""))
if "CC BY 4.0" not in license_name:
    raise RuntimeError(f"Refusing to train. data.yaml license is {license_name!r}, expected CC BY 4.0.")
loaded["path"] = str(dataset_dir.resolve())
data_yaml.write_text(yaml.safe_dump(loaded, sort_keys=False, allow_unicode=True), encoding="utf-8")
print(f"License: {license_name}")
print(f"data yaml: {data_yaml}")


## 續訓

`last.pt` 和 `best.pt` 會留在 `/kaggle/working`。Save Version 之後，下一個筆記本 Add Input，把 `RESUME_LAST` 指到唯讀的 `last.pt`。筆記本會先複製到可寫目錄，再用 `resume=True`。棋子圖在暫存碟上，下一輪會再從 Hub 下載，不需要帳號。

### English

Save the notebook output as a Kaggle Dataset, set `RESUME_LAST` to that `last.pt`, and continue with `resume=True`. The chess images are scratch and are downloaded again from the public Hub repo.


In [ ]:
import sys
from pathlib import Path

repo = Path(REPO_ROOT)
module = repo / "src" / "detect" / "coco_early_stop.py"
if not module.is_file():
    raise FileNotFoundError(
        "Clone this repository to REPO_ROOT so src/detect/coco_early_stop.py can be imported. "
        "Do not put a Kaggle or Roboflow token in the notebook."
    )
sys.path.insert(0, str(repo))
from src.detect.coco_early_stop import AcceptableResultStop, StopConfig

print(f"stop rules: {module}")


In [ ]:
import shutil
import time
from pathlib import Path

from ultralytics import YOLO
from ultralytics.utils import SETTINGS

SETTINGS["sync"] = False

resume_weights = None
if RESUME_LAST:
    source = Path(RESUME_LAST)
    if not source.is_file():
        raise FileNotFoundError(source)
    resume_weights = Path("/kaggle/working/chess-yolo11/resume/last.pt")
    resume_weights.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(source, resume_weights)
    print(f"Copied {source} to {resume_weights}")

stopper = AcceptableResultStop(
    StopConfig(
        loss_min_delta=LOSS_MIN_DELTA,
        loss_patience=LOSS_PATIENCE,
        map_target=MAP_TARGET,
        time_budget_hours=TIME_HOURS,
    ),
    started_at=time.time(),
)
model = YOLO(str(resume_weights) if resume_weights is not None else MODEL)
model.add_callback("on_fit_epoch_end", stopper.on_fit_epoch_end)

save_dir = str(Path(PROJECT) / "train")
if resume_weights is not None:
    model.train(
        resume=True,
        data=str(data_yaml),
        imgsz=IMGSZ,
        batch=BATCH,
        device=DEVICE,
        workers=WORKERS,
        patience=PATIENCE,
        save_period=SAVE_PERIOD,
        time=TIME_HOURS,
        save_dir=save_dir,
        exist_ok=True,
    )
else:
    model.train(
        data=str(data_yaml),
        epochs=EPOCHS,
        imgsz=IMGSZ,
        batch=BATCH,
        device=DEVICE,
        workers=WORKERS,
        patience=PATIENCE,
        amp=AMP,
        pretrained=PRETRAINED,
        project=PROJECT,
        name="train",
        exist_ok=True,
        save=True,
        save_period=SAVE_PERIOD,
        time=TIME_HOURS,
        cache=False,
        plots=False,
        val=True,
        seed=0,
    )


In [ ]:
import shutil
from pathlib import Path

weights = Path(PROJECT) / "train" / "weights"
last = weights / "last.pt"
best = weights / "best.pt"
missing = [path for path in (last, best) if not path.is_file()]
if missing:
    raise FileNotFoundError(f"Missing checkpoints: {missing}")
shutil.copy2(last, Path("/kaggle/working/last.pt"))
shutil.copy2(best, Path("/kaggle/working/best.pt"))
print("Kept /kaggle/working/last.pt and /kaggle/working/best.pt")
if stopper.last_decision is not None and stopper.last_decision.stop:
    print(f"Stopped early: {stopper.last_decision.reason}")


## 自動停止

`PATIENCE = 15` 交給 Ultralytics。8.4 的 fitness 是驗證 mAP50-95。`on_fit_epoch_end` 另外在損失平台、mAP50-95 達到 `MAP_TARGET`、或超過 `TIME_HOURS` 時把 `trainer.stop` 設成 True。這發生在該輪驗證和存檔之後，所以 `best.pt` 和 `last.pt` 會留下。

雙卡時只有 rank 0 做決定。Ultralytics 8.4 會廣播 `trainer.stop`。若新版本不再廣播，把 `DEVICE` 改成 `"0"`，或只靠 `patience`。

### English

Same stop rules as the COCO notebook: Ultralytics patience plus `AcceptableResultStop`. Rank 0 decides on two GPUs. If the DDP broadcast fails, use `device=0` or patience only.
